In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
import numpy.linalg as la
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.metrics import average_precision_score,precision_recall_curve
from sklearn.utils.random import sample_without_replacement
import tensorflow as tf
import matplotlib.pyplot as plt
sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_elastic import NMF_logistic

from scipy.optimize import nnls
from tqdm import trange

In [ ]:
tf.__version__

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)

indx_pos = (behaviornon1==1)&(condition==4)
indx_neg1 = (behaviornon1==2)&(condition==4)
indx_neg2 = (behaviornon1==2)&(condition==6)
indx_neg3 = (behaviornon1==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos

In [ ]:
y = np.zeros(N)
y[indx_pos] = 1

mouse = mouse[indx_tot]
group = group[indx_tot]
expDate = expDate[indx_tot]
behavior = behavior[indx_tot]
behaviornon1 = behaviornon1[indx_tot]
time = time[indx_tot]
condition = condition[indx_tot]
y = y[indx_tot]

N = len(mouse)

training_set_idx = np.ones(N)
training_set_idx[mouse=='Mouse048'] = 0
training_set_idx[mouse=='Mouse7980'] = 0
training_set_idx[mouse=='Mouse7998'] = 0

granger = np.exp(granger)
granger[granger>10] = 10from tqdm import trange
power = power*10
power[power>6] = 6

X = np.hstack(def compute_optimal_W(X, S):
    # Compute the optimal W
    W = X @ S.T @ la.inv(S @ S.T)
    return W(power,coherence,granger))
X = X[indx_tot]

In [ ]:
X_train = X[training_set_idx==1]
m_train = mouse[training_set_idx==1]
y_train = y[training_set_idx==1]

X_test = X[training_set_idx==0]
m_test = mouse[training_set_idx==0]
y_test = y[training_set_idx==0]

mu = 1.0
#>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
# Add the newer data
power,coherence,granger,labels_new = load_data('/media/austin/ThickBoy__1/DataAgression_Granger2/CL_baseline_all_validate3.mat',fBounds=(1,56),feature_list=['power','coherence','granger'])

power = 10*power
power = power.astype(np.float32)
power[power>6] = 6

coherence = coherence.astype(np.float32)
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

X_new = np.hstack((power,coherence,granger))

windows_new = labels_new['windows']
mouse_new = np.squeeze(windows_new['mouse'])
expDate_new = np.squeeze(windows_new['expDate'])
group_new = np.squeeze(windows_new['group'])
condition_new = np.squeeze(windows_new['condition'])
behavior_new = np.squeeze(windows_new['behavior'])
time_new = np.squeeze(windows_new['time'])


In [ ]:
idx_pos_new = (condition_new==4)&(behavior_new==1)
indx_neg_new = (behavior_new==2)&((condition_new==4)|(condition_new==6)|(condition_new==8))
y_new = np.zeros(len(mouse_new))
y_new[idx_pos_new] = 1
idx_tot_new = idx_pos_new|indx_neg_new

X_new = X_new[idx_tot_new]
mouse_new = mouse_new[idx_tot_new]
y_new = y_new[idx_tot_new]


mice_new = np.unique(mouse_new)
nMice = len(mice_new)
mice_new_train = mice_new[:4]


ids = np.zeros(len(mouse_new))
for i in range(4):
    ids[mouse_new==mice_new_train[i]] = 1

print('>>>>>>>>>>>>>.')
print(y_new.shape)
print(ids.shape)
print(mouse_new.shape)
print(X_new.shape)

X_train_new = X_new[ids==1,:]
X_test_new = X_new[ids==0,:]
y_train_new = y_new[ids==1]
y_test_new = y_new[ids==0]


In [ ]:
mm_train_new = mouse_new[ids==1]
mm_test_new = mouse_new[ids==0]

In [ ]:
X_train_tot = np.vstack((X_train,X_train_new))
y_train_tot = np.concatenate((y_train,y_train_new))
weights_g = np.ones(X_train_tot.shape[0])
weights_s = np.ones(X_train_tot.shape[0])


In [ ]:
mm_train_new.shape

In [ ]:
m_train_tot = np.concatenate((m_train,mm_train_new))
m_train_tot.shape

In [ ]:
with open('Unbalanced_Elastic_12_enc_1.0.p','rb') as f:
    myDict = pickle.load(f)

In [ ]:
myDict.keys()

In [ ]:
components_estimated = myDict['components']

In [ ]:
components_estimated.shape

In [ ]:
S_estimated = np.vstack((myDict['S_train'],myDict['S_train_new']))

In [ ]:
S_estimated.shape

In [ ]:
X_train_tot.shape

In [ ]:
X_recon_non = np.dot(S_estimated[:,1:],components_estimated[1:])

In [ ]:
X_diff = X_train_tot-X_recon_non

In [ ]:
np.mean(X_diff**2)

In [ ]:
np.mean(X_train_tot**2)

In [ ]:
def compute_nonnegative_W(X, S):
    # Ensure S is a column vector
    S = S[:, np.newaxis]  # Shape becomes (15211, 1)
    
    # Initialize W with the appropriate shape
    W_opt = np.zeros(X.shape[1])  # Shape (9856,)
    
    # Solve NNLS for each column of X
    for i in range(X.shape[1]):
        W_opt[i], _ = nnls(S, X[:, i])
    
    return W_opt

In [ ]:
def bootstrap_W(X, S, n_rep=1000):
    """Perform bootstrap to estimate confidence intervals for W."""
    n_samples = X.shape[0]  # Number of rows in X
    
    # Store W estimates from each bootstrap iteration
    W_estimates = np.zeros((n_rep, X.shape[1]))  # Shape (n_rep, 9856)
    
    # Bootstrap loop
    for i in trange(n_rep):
        # Resample rows of X and corresponding entries of S with replacement
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        X_resampled = X[indices]
        S_resampled = S[indices]
        
        # Compute W_opt for the resampled data
        W_estimates[i, :] = compute_nonnegative_W(X_resampled, S_resampled)
    
    return W_estimates

In [ ]:
W_opt = compute_nonnegative_W(X_diff,S_estimated[:,0])

In [ ]:
W_estimates = bootstrap_W(X_diff,S_estimated[:,0])

In [ ]:
f

In [ ]:
quantiles_lower = np.quantile(W_estimates,.025,axis=0)
quantiles_upper = np.quantile(W_estimates,.025,axis=0)

In [ ]:
plt.hist(W_opt,50);

In [ ]:
plt.scatter(W_opt,components_estimated[0])

In [ ]:
labels.keys()

In [ ]:
features = np.concatenate((labels['powerFeatures'],labels['cohFeatures'],labels['gcFeatures']))

In [ ]:
features[-10]

In [ ]:
np.where(features=='PL->NAc 20')

In [ ]:
np.where(features=='PL->MeA 20')

In [ ]:
labels['area']

In [ ]:
np.where(features=='PL->LHb 20')

In [ ]:
from scipy.stats import wilcoxon

In [ ]:
stat,pval = wilcoxon(W_estimates[:,8363],W_estimates[:,7803])
print('NAC vs MEA:',stat,pval)

In [ ]:
stat,pval = wilcoxon(W_estimates[:,8363],W_estimates[:,5451])
print(stat,pval)
print('NAC vs LHb:',stat,pval)

In [ ]:
stat,pval = wilcoxon(W_estimates[:,7803],W_estimates[:,5451])
print(stat,pval)
print('MEA vs LHb:',stat,pval)

In [ ]:
def bootstrap_W_new(X, S,m,n_rep=20):
    """Perform bootstrap to estimate confidence intervals for W."""
    n_samples = X.shape[0]  # Number of rows in X
    m_unique = np.unique(m)
    nm = len(m_unique)
    
    # Store W estimates from each bootstrap iteration
    W_estimates = np.zeros((n_rep, X.shape[1]))  # Shape (n_rep, 9856)
    
    # Bootstrap loop
    for i in trange(n_rep):
        # Resample rows of X and corresponding entries of S with replacement
        sampled_labels = np.random.choice(m_unique, size=nm, replace=True)
        indices = np.isin(m, sampled_labels)
        
        X_resampled = X[indices]
        S_resampled = S[indices]
        print(X_resampled.shape)
        
        # Compute W_opt for the resampled data
        W_estimates[i, :] = compute_nonnegative_W(X_resampled, S_resampled)
    
    return W_estimates

In [ ]:
W_estimates_new = bootstrap_W_new(X_diff,S_estimated[:,0],m_train_tot)

In [ ]:
stat,pval = wilcoxon(W_estimates_new[:,8363],W_estimates_new[:,7803])
print('NAC vs MEA:',stat,pval)

In [ ]:
stat,pval = wilcoxon(W_estimates_new[:,8363],W_estimates_new[:,5451])
print(stat,pval)
print('NAC vs LHb:',stat,pval)

In [ ]:
stat,pval = wilcoxon(W_estimates_new[:,7803],W_estimates_new[:,5451])
print(stat,pval)
print('MEA vs LHb:',stat,pval)

In [ ]:
W_estimates_new[:,8363]

In [ ]:
W_estimates_new[:,7803]

In [ ]:
W_estimates_new[:,5451]